This notebook takes the NAICS Employed, Quarterly, and Monthly Vacancies of Canada and its provinces/territories from the S3 folder, checks for nulls, dupes, schema and clean/finalizes them, then transforms them into pivot tables to prepare for transformation in dbt. Finally, exporting the pivot tables and cleaned NAICS Employed back into S3

In [0]:
from pyspark.sql.functions import col, split, trim, sum as spark_sum, when, to_date

from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("base_input_path", "s3a://your-bucket-name/processed/")  # please change to your own bucket
dbutils.widgets.text("base_output_path", "s3a://your-bucket-name/curated/")  # please change to your own bucket

base_input = dbutils.widgets.get("base_input_path").rstrip("/")
base_output = dbutils.widgets.get("base_output_path").rstrip("/")

df_Monthly  = spark.read.parquet(f"{base_input}/monthly_vacancies_processed/")
df_NAICS    = spark.read.parquet(f"{base_input}/naics_employed_processed/")
df_Quarterly = spark.read.parquet(f"{base_input}/quarterly_vacancies_processed/")

In [0]:
# Validation of loaded Raw data
for name, df in [("Monthly", df_Monthly), ("Quarterly", df_Quarterly), ("NAICS", df_NAICS)]:
    row_count = df.count()
    assert row_count > 0, f"{name} loaded empty — check S3 path"
    print(f"{name}: {row_count} rows loaded")

<h1>Cleaning Data</h1>

<h2>Monthly</h2>

In [0]:
df_Monthly.printSchema()
display(df_Monthly.limit(20))
display(df_Monthly.select("Date").distinct().orderBy("Date").limit(200))
display(df_Monthly.select("Statistics").distinct().orderBy("Statistics").limit(5))

In [0]:
Monthly_row_count = df_Monthly.count()

print("total rows:", Monthly_row_count)

# null counts per column
nulls_Monthly = df_Monthly.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_Monthly.columns])
display(nulls_Monthly)

print("distinct GEO:", df_Monthly.select("GEO").distinct().count())
print("distinct Year:", df_Monthly.select("Year").distinct().count())
print("distinct Statistics:", df_Monthly.select("Statistics").distinct().count())

In [0]:
# duplicates by GEO+Year+Statistics
dupes1_Monthly = df_Monthly.groupBy("GEO","Year","Statistics") \
                  .count().filter("count > 1").orderBy("count", ascending=True) # Ascending to find what year has the most missing values
display(dupes1_Monthly.limit(200))
print("dup groups (GEO,Year,Statistics):", dupes1_Monthly.count())

# duplicates by GEO+Date+Statistics
dupes2_Monthly = df_Monthly.groupBy("GEO","Date","Statistics").count().filter("count > 1").orderBy("count", ascending=True)
display(dupes2_Monthly.limit(20))
print("dup groups (GEO,Date,Statistics):", dupes2_Monthly.count())


In [0]:
clean_Monthly = df_Monthly.withColumn("Date_parsed", to_date(col("Date"), "yyyy-MM"))
display(clean_Monthly.select("Date", "Date_parsed").distinct().orderBy("Date").limit(50))
display(clean_Monthly.select("Date","Date_parsed").limit(50))

<h2>Quarterly</h2>

In [0]:
df_Quarterly.printSchema()
display(df_Quarterly.limit(20))
display(df_Quarterly.select("Date").distinct().orderBy("Date").limit(200))
display(df_Quarterly.select("Statistics").distinct().orderBy("Statistics").limit(5))
display(df_Quarterly.select("NAICS").distinct().orderBy("NAICS").limit(10))

In [0]:
Quarterly_row_count = df_Quarterly.count()

print("total rows:", Quarterly_row_count)

# null counts per column
nulls_Quarterly = df_Quarterly.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_Quarterly.columns])
display(nulls_Quarterly)

print("distinct GEO:", df_Quarterly.select("GEO").distinct().count())
print("distinct Year:", df_Quarterly.select("Year").distinct().count())
print("distinct Statistics:", df_Quarterly.select("Statistics").distinct().count())
print("distinct NAICS:", df_Quarterly.select("NAICS").distinct().count())

In [0]:
# duplicates by GEO+Year+Statistics+NAICS
dupes1_Quarterly = df_Quarterly.groupBy("GEO","Year","Statistics","NAICS") \
                  .count().filter("count > 1").orderBy("count", ascending=True) # Ascending to find what year has the most missing values
display(dupes1_Quarterly.limit(500))
print("dup groups (GEO,Year,Statistics):", dupes1_Quarterly.count())

# duplicates by GEO+Date+Statistics+NAICS
dupes2_Quarterly = df_Quarterly.groupBy("GEO","Date","Statistics","NAICS").count().filter("count > 1").orderBy("count", ascending=True)
display(dupes2_Quarterly.limit(20))
print("dup groups (GEO,Date,Statistics):", dupes2_Quarterly.count())


In [0]:
clean_Quarterly = df_Quarterly.withColumn("Date_parsed", to_date(col("Date"), "yyyy-MM"))
display(clean_Quarterly.select("Date", "Date_parsed").distinct().orderBy("Date").limit(50))
display(clean_Quarterly.select("Date","Date_parsed").limit(50))

<h2>NAICS</h2>

In [0]:
df_NAICS.printSchema()
display(df_NAICS.limit(20))
display(df_NAICS.select("Date").distinct().orderBy("Date").limit(200))
display(df_NAICS.select("NAICS").distinct().orderBy("NAICS").limit(5))

In [0]:
NAICS_row_count = df_NAICS.count()

print("total rows:", NAICS_row_count)

# null counts per column
nulls_NAICS = df_NAICS.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_NAICS.columns])
display(nulls_NAICS)

print("distinct GEO:", df_NAICS.select("GEO").distinct().count())
print("distinct Year:", df_NAICS.select("Year").distinct().count())
print("distinct NAICS:", df_NAICS.select("NAICS").distinct().count())

In [0]:
# duplicates by GEO+Year+Statistics
dupes1_NAICS = df_NAICS.groupBy("GEO","Year","NAICS") \
                  .count().filter("count > 1").orderBy("count", ascending=True) # Ascending to find what year has the most missing values
print("dup groups (GEO,Year,Statistics):", dupes1_NAICS.count())

# duplicates by GEO+Date+Statistics
dupes2_NAICS = df_NAICS.groupBy("GEO","Date","NAICS").count().filter("count > 1").orderBy("count", ascending=True)
display(dupes2_NAICS.limit(20))
print("dup groups (GEO,Date,Statistics):", dupes2_NAICS.count())


In [0]:
clean_NAICS = df_NAICS.withColumn("Date_parsed", to_date(col("Date"), "yyyy-MM"))
display(clean_NAICS.select("Date", "Date_parsed").distinct().orderBy("Date").limit(20))
display(clean_NAICS.select("Date","Date_parsed").limit(20))
display(clean_NAICS.head(10))

<h1>Pivoting</h1>

<h2>Monthly vacancies pivot</h2>

In [0]:
pivoted_Monthly_Vac = clean_Monthly.groupBy("Year","Date_parsed", "GEO") \
    .pivot("Statistics") \
    .sum("VALUE")

In [0]:
# Checking for dupes
pivoted_Monthly_Vac.groupBy(
    "Date_parsed", "GEO", "Year"
).count().filter("count > 1").show()

In [0]:
pivoted_Monthly_Vac.orderBy("Date_parsed").show(10)

In [0]:
# Checking for nulls in each column
display(pivoted_Monthly_Vac.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in pivoted_Monthly_Vac.columns]))

<h2>NAICS employed pivot</h2>

In [0]:
pivoted_NAICS = (
    clean_NAICS .select("Year", "Date_parsed", "NAICS", "GEO", "VALUE")
        .withColumnRenamed("VALUE", "Total Employment")
)

In [0]:
display(pivoted_NAICS)
pivoted_NAICS.count()

<h2>Quarterly vacancies pivot</h2>

In [0]:
pivoted_Quarterly_Vac = clean_Quarterly.groupBy("Year","Date_parsed", "GEO", "NAICS") \
    .pivot("Statistics") \
    .sum("VALUE")

In [0]:
pivoted_Quarterly_Vac.groupBy("Date_parsed","GEO","NAICS","Year") \
    .count().filter("count > 1").show(50, truncate=False)

In [0]:
display(pivoted_Quarterly_Vac.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in pivoted_Quarterly_Vac.columns]))

In [0]:
display(pivoted_Quarterly_Vac.orderBy("NAICS", "Date_parsed", "GEO"))

<h1>Exporting to S3</h1>

In [0]:
# Validation before export
def validate_export_ready(df, name, required_cols, critical_cols, strict=False):
    actual_cols = df.columns
    missing = [c for c in required_cols if c not in actual_cols]
    assert not missing, f"{name} missing columns before export: {missing}"
    if strict:
        extra = [c for c in actual_cols if c not in required_cols]
        assert not extra, f"{name} has unexpected extra columns: {extra}"
        print(f"{name} schema OK (strict) - exact columns match")
    else:
        print(f"{name} schema OK - required columns present")

    nulls = df.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) 
                                    for c in df.columns]).collect()[0].asDict()

    for c in critical_cols:
        assert nulls[c] == 0, f"Nulls found in critical column: {c} ({nulls[c]} nulls)"
    print(f"{name}: No nulls in critical columns")

    dupes = df.groupBy(critical_cols) \
                  .count().filter("count > 1").orderBy("count", ascending=True)

    assert dupes.count() == 0, f"Duplicate records found: {dupes.show(truncate=False)}"
    print(f"{name}: No duplicates found")


# NAICS
NAICS_required_cols = ["Year", "GEO", "NAICS", "Date_parsed", "Total Employment"]
NAICS_critical_cols = ["Year", "GEO", "NAICS", "Date_parsed"]

validate_export_ready(pivoted_NAICS, "NAICS Employed", NAICS_required_cols, NAICS_critical_cols, strict=True) 

assert NAICS_row_count == pivoted_NAICS.count(), "NAICS row count is not what's expected"
print(f"NAICS row count is what's expected\n")

# Monthly Vac
pivoted_Monthly_required_cols = ["Year", "GEO", "Date_parsed", "Job vacancies", "Job vacancy rate", "Payroll employees"]
pivoted_Monthly_critical_cols = ["Year", "GEO", "Date_parsed"]

validate_export_ready(pivoted_Monthly_Vac, "Monthly pivot", pivoted_Monthly_required_cols, pivoted_Monthly_critical_cols, strict=True)

assert Monthly_row_count/3 == pivoted_Monthly_Vac.count(), "Monthly row count is not what's expected"
print(f"Monthly row count is what's expected\n")

# Quarterly Vac
pivoted_Quarterly_Vac_required_cols = ["Year", "GEO", "NAICS", "Date_parsed", "Average offered hourly wage","Job vacancies", "Job vacancy rate", "Payroll employees"]
pivoted_Quarterly_Vac_critical_cols = ["Year", "GEO", "NAICS", "Date_parsed"]

validate_export_ready(pivoted_Quarterly_Vac, "Quarterly pivot", pivoted_Quarterly_Vac_required_cols, pivoted_Quarterly_Vac_critical_cols, strict=True)

assert pivoted_Quarterly_Vac.count() == (Quarterly_row_count/4), "Quarterly row count is not what's expected"
print(f"Quarterly row count is what's expected\n")

print("All dataframes validated - ready to export to S3")

In [0]:
# Pivoted NAICS with the Schema
pivoted_NAICS.write.mode("overwrite").partitionBy("Year", "GEO").parquet(f"{base_output}/naics_employed_curated/")

# Monthly Vac and Employed
pivoted_Monthly_Vac.write.mode("overwrite").partitionBy("Year", "GEO").parquet(f"{base_output}/monthly_vacancies_curated/")

# Quarterly Vac and Employed
pivoted_Quarterly_Vac.write.mode("overwrite").partitionBy("Year", "GEO").parquet(f"{base_output}/quarterly_vacancies_curated/")